In [1]:
# =============================================================================
# SSL ENCODER DIAGNOSTIC – FULL DATASET (with multi‑hot evaluation)
# =============================================================================

import os
import pickle
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Batch
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    f1_score, roc_auc_score, recall_score, precision_score,
    classification_report, average_precision_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ------------------------- CONFIGURATION -------------------------
DATA_DIR = "GothamDataset2025/processed_features/gotham_graphs"
CHECKPOINT_DIR = os.path.join(DATA_DIR, "ssl_checkpoints")
ENCODER_PATH = os.path.join(CHECKPOINT_DIR, "ssl_encoder_best.pth")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

# ------------------------- LOAD GRAPHS -------------------------
def load_graphs(split):
    with open(os.path.join(DATA_DIR, f"{split}_graphs.pkl"), 'rb') as f:
        return pickle.load(f)

train_graphs = load_graphs('train')
val_graphs   = load_graphs('val')
test_graphs  = load_graphs('test')
all_graphs = train_graphs + val_graphs + test_graphs
print(f"Total graphs: {len(all_graphs)} (Train={len(train_graphs)}, Val={len(val_graphs)}, Test={len(test_graphs)})")

# ------------------------- UNIFIED CLASS MAPPING (from training) -------------------------
def get_class_mapping(graphs):
    attack_names = set()
    for g in graphs:
        for name in g.edge_attack_names:
            if name != 'Benign':
                attack_names.add(name)
    attack_names = sorted(list(attack_names))
    class_to_idx = {name: i for i, name in enumerate(attack_names)}
    num_classes = len(attack_names)
    return attack_names, class_to_idx, num_classes

attack_names, class_to_idx, num_classes = get_class_mapping(train_graphs)
print(f"Attack classes ({num_classes}): {attack_names}")

# ------------------------- ASSIGN LABELS TO ALL GRAPHS (multi‑hot) -------------------------
def assign_labels(graphs, class_to_idx, num_classes):
    for g in graphs:
        if not hasattr(g, 'y_binary'):
            g.y_binary = torch.tensor(
                1 if any(name != 'Benign' for name in g.edge_attack_names) else 0,
                dtype=torch.float
            ).unsqueeze(0)
        if not hasattr(g, 'y_multihot'):
            labels = torch.zeros(num_classes, dtype=torch.float)
            for name in g.edge_attack_names:
                if name != 'Benign' and name in class_to_idx:
                    labels[class_to_idx[name]] = 1.0
            g.y_multihot = labels.unsqueeze(0)

assign_labels(all_graphs, class_to_idx, num_classes)

# Build binary labels and multi‑hot labels
binary_labels = []
multi_hot_labels = []
for g in all_graphs:
    binary_labels.append(g.y_binary.item())
    multi_hot_labels.append(g.y_multihot.squeeze().numpy())
binary_labels = np.array(binary_labels)
multi_hot_labels = np.vstack(multi_hot_labels)

print("\nEdge attack counts (total across all graphs):")
edge_counts = {}
for g in all_graphs:
    for name in g.edge_attack_names:
        edge_counts[name] = edge_counts.get(name, 0) + 1
for name, cnt in sorted(edge_counts.items(), key=lambda x: -x[1]):
    print(f"  {name}: {cnt}")

# ------------------------- DEFINE GAT ENCODER -------------------------
class GATEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_heads, num_layers, edge_dim=0, dropout=0.0):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.dropout = dropout
        self.edge_dim = edge_dim
        self.convs.append(GATConv(in_channels, hidden_channels, heads=num_heads,
                                  edge_dim=edge_dim, concat=True, dropout=dropout))
        self.bns.append(nn.BatchNorm1d(hidden_channels * num_heads))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_channels * num_heads, hidden_channels, heads=num_heads,
                                      edge_dim=edge_dim, concat=True, dropout=dropout))
            self.bns.append(nn.BatchNorm1d(hidden_channels * num_heads))
        self.convs.append(GATConv(hidden_channels * num_heads, hidden_channels, heads=1,
                                  edge_dim=edge_dim, concat=False, dropout=dropout))
        self.bns.append(nn.BatchNorm1d(hidden_channels))

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        x = torch.nan_to_num(x, nan=0.0, posinf=1e3, neginf=-1e3)
        if edge_attr is not None:
            edge_attr = torch.nan_to_num(edge_attr, nan=0.0, posinf=1e3, neginf=-1e3)
        for i, conv in enumerate(self.convs):
            if self.edge_dim > 0 and edge_attr is not None:
                x = conv(x, edge_index, edge_attr=edge_attr)
            else:
                x = conv(x, edge_index)
            x = self.bns[i](x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return torch.nan_to_num(x, nan=0.0, posinf=1e3, neginf=-1e3)

# ------------------------- LOAD PRETRAINED ENCODER -------------------------
sample = all_graphs[0]
node_dim = sample.x.size(1)
edge_dim = sample.edge_attr.size(1) if sample.edge_attr is not None else 0

encoder = GATEncoder(node_dim, hidden_channels=128, num_heads=4, num_layers=3,
                     edge_dim=edge_dim, dropout=0.2).to(DEVICE)
encoder.load_state_dict(torch.load(ENCODER_PATH, map_location=DEVICE))
encoder.eval()
print(f"\nLoaded SSL encoder from {ENCODER_PATH}")

# ------------------------- EXTRACT EMBEDDINGS -------------------------
embeddings = []
batch_size = 2
for i in tqdm(range(0, len(all_graphs), batch_size), desc="Extracting embeddings"):
    batch_graphs = all_graphs[i:i+batch_size]
    bg = Batch.from_data_list(batch_graphs).to(DEVICE)
    with torch.no_grad():
        emb = encoder(bg.x, bg.edge_index, bg.edge_attr, bg.batch)
    embeddings.append(emb.cpu().numpy())
embeddings = np.vstack(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

# Shuffle and split (no stratification)
X, y_bin, y_multi = shuffle(embeddings, binary_labels, multi_hot_labels, random_state=42)
split_idx = int(0.7 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_bin_train, y_bin_test = y_bin[:split_idx], y_bin[split_idx:]
y_multi_train, y_multi_test = y_multi[:split_idx], y_multi[split_idx:]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

# Scale embeddings
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------- BINARY CLASSIFICATION -------------------------
clf_bin = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_bin.fit(X_train_scaled, y_bin_train)
y_bin_pred = clf_bin.predict(X_test_scaled)
y_bin_prob = clf_bin.predict_proba(X_test_scaled)[:, 1]
bin_f1 = f1_score(y_bin_test, y_bin_pred)
bin_auc = roc_auc_score(y_bin_test, y_bin_prob)

print("\n=== BINARY CLASSIFICATION (Benign vs Attack) ===")
print(f"F1: {bin_f1:.4f}")
print(f"AUC: {bin_auc:.4f}")

# ------------------------- MULTI‑LABEL (PER‑CLASS) EVALUATION -------------------------
print("\n=== MULTI‑LABEL PER‑CLASS PERFORMANCE (One‑vs‑Rest) ===")
print(f"{'Class':20} {'F1':8} {'AUC':8} {'PR-AUC':8} {'Recall':8} {'Precision':8}")

per_class_f1 = {}
per_class_auc = {}
per_class_pr_auc = {}
per_class_recall = {}
per_class_precision = {}

for c in range(num_classes):
    y_c_test = y_multi_test[:, c]
    # Skip if class has too few samples (less than 5 in test)
    if np.sum(y_c_test) < 5:
        name = attack_names[c]
        print(f"{name:20} {'(too few samples)'}")
        continue
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_train_scaled, y_multi_train[:, c])
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]
    f1 = f1_score(y_c_test, y_pred)
    auc = roc_auc_score(y_c_test, y_prob)
    pr_auc = average_precision_score(y_c_test, y_prob)
    recall = recall_score(y_c_test, y_pred)
    precision = precision_score(y_c_test, y_pred)
    name = attack_names[c]
    per_class_f1[name] = f1
    per_class_auc[name] = auc
    per_class_pr_auc[name] = pr_auc
    per_class_recall[name] = recall
    per_class_precision[name] = precision
    print(f"{name:20} {f1:8.4f} {auc:8.4f} {pr_auc:8.4f} {recall:8.4f} {precision:8.4f}")

# ------------------------- SUMMARY -------------------------
print("\n" + "="*50)
print("DIAGNOSTIC SUMMARY")
print("="*50)
if bin_auc > 0.85:
    print(" SSL encoder is good for binary classification.")
else:
    print("❌ Binary AUC < 0.85 – improve SSL.")

# Compute macro average over classes with enough samples
valid_classes = [k for k, v in per_class_auc.items() if v > 0]
if valid_classes:
    avg_f1 = np.mean([per_class_f1[c] for c in valid_classes])
    avg_auc = np.mean([per_class_auc[c] for c in valid_classes])
    print(f"\nAverage over {len(valid_classes)} classes with >5 samples:")
    print(f"  Macro F1: {avg_f1:.4f}")
    print(f"  Macro AUC: {avg_auc:.4f}")
else:
    print("\n⚠️ Not enough samples in any class for reliable multi‑label metrics.")
print("="*50)

# Save results
results = {
    'binary_auc': float(bin_auc),
    'binary_f1': float(bin_f1),
    'per_class': {k: {'f1': per_class_f1.get(k, 0), 'auc': per_class_auc.get(k, 0),
                     'pr_auc': per_class_pr_auc.get(k, 0)} for k in set(per_class_f1.keys()) | set(per_class_auc.keys())}
}
with open('ssl_diagnostic_multilabel.json', 'w') as f:
    json.dump(results, f, indent=4)
print("\nDiagnostic results saved to ssl_diagnostic_multilabel.json")

Using device: cuda
Total graphs: 1332 (Train=932, Val=200, Test=200)
Attack classes (14): ['C&C Communication', 'CoAP Amplification', 'File Download', 'Ingress Tool Transfer', 'Merlin C&C Communication', 'Merlin ICMP Flooding', 'Merlin TCP Flooding', 'Merlin UDP Flooding', 'Mirai C&C Communication', 'Reporting', 'TCP Scan', 'Telnet Brute Force', 'UDP Scan', 'Unknown']

Edge attack counts (total across all graphs):
  Mirai TCP Flooding: 9546522
  Mirai UDP Flooding: 4879814
  TCP Scan: 321676
  Benign: 265744
  Merlin TCP Flooding: 120011
  Merlin UDP Flooding: 30000
  Telnet Brute Force: 21831
  Unknown: 9140
  UDP Scan: 4444
  Merlin C&C Communication: 848
  Mirai C&C Communication: 458
  Reporting: 192
  C&C Communication: 188
  Ingress Tool Transfer: 176
  File Download: 20
  Merlin ICMP Flooding: 18
  Mirai GRE Flooding: 15
  CoAP Amplification: 4

Loaded SSL encoder from GothamDataset2025/processed_features/gotham_graphs\ssl_checkpoints\ssl_encoder_best.pth


Extracting embeddings: 100%|██████████| 666/666 [00:52<00:00, 12.66it/s] 


Embeddings shape: (1332, 128)
Train size: 932, Test size: 400

=== BINARY CLASSIFICATION (Benign vs Attack) ===
F1: 0.8672
AUC: 0.9807

=== MULTI‑LABEL PER‑CLASS PERFORMANCE (One‑vs‑Rest) ===
Class                F1       AUC      PR-AUC   Recall   Precision
C&C Communication      0.5143   0.9687   0.2730   1.0000   0.3462
CoAP Amplification   (too few samples)
File Download        (too few samples)
Ingress Tool Transfer   0.4571   0.9997   0.9861   1.0000   0.2963
Merlin C&C Communication   0.8615   0.9465   0.8603   0.7778   0.9655
Merlin ICMP Flooding   0.0606   0.4978   0.1715   0.1250   0.0400
Merlin TCP Flooding  (too few samples)
Merlin UDP Flooding  (too few samples)
Mirai C&C Communication   0.6703   0.9059   0.5835   0.9385   0.5214
Reporting              0.4571   0.9997   0.9861   1.0000   0.2963
TCP Scan               0.5143   0.9687   0.2730   1.0000   0.3462
Telnet Brute Force     0.5217   0.8410   0.6638   0.8000   0.3871
UDP Scan               0.9032   0.9976   0.8701  